# 07 - Análisis de la partición train/test
En este notebook vamos a comprobar cómo dividir los conjuntos de datos en train y test (`urban_accessibility_ml/modeling/split.py`), usando el conjunto de datos unificado (`data/processed/streets_unified.csv`).

Si los tramos de una misma calle se parecen entre sí, repartirlos entre train y test produciría fuga de información. Vamos a comprobar primero que esa relación existe y luego se comparan dos opciones de agrupación (`group_col`):
* La calle (`nombre_calle_base`).
* El barrio (`barrio`).

Se evaluará si cada fold tras realizar el split contenga tramos de la clase A, tanto en entrenamiento como en test.

## Carga de datos

In [ ]:
import pandas as pd

from urban_accessibility_ml.modeling.split import (
    BARRIO_COL, CITY_COL, CLASS_COL, ID_COL, STREET_COL,
    split_cross_city, split_mixed, split_within_city,
)

from urban_accessibility_ml.config import PROCESSED_DATA_DIR

df = pd.read_csv(PROCESSED_DATA_DIR / "streets_unified.csv")
CITIES = ["Algeciras", "Albacete"]
class_of = df.set_index(ID_COL)[CLASS_COL]

## Grupos por ciudad
Comprobamos en el conjunto de datos unificados, cuántos tramos hay por ciudad, cuántas calles, cuántos barrios, cuántos tramos son de clase A, y finalmente cuántos barrios tienen al menos un tramo de clase A.

In [ ]:
df.groupby(CITY_COL).agg(
    tramos=(ID_COL, "size"),
    calles=(STREET_COL, "nunique"),
    barrios=(BARRIO_COL, "nunique"),
    tramos_A=(CLASS_COL, lambda s: (s == "A").sum()),
    barrios_con_A=(BARRIO_COL, lambda s: s[df.loc[s.index, CLASS_COL] == "A"].nunique()),
)

Existen muy pocos barrios con tramos A, especialmente en Albacete (2 de 27), no tanto en Algeciras (11 de 44).
Comprobamos ahora cuántos tramos hay en cada calle, y en cada barrio.

In [ ]:
pd.DataFrame({
    f"{group_col}": df.groupby(CITY_COL)[group_col].apply(lambda s: s.value_counts().describe()[["min", "50%", "max"]])
    for group_col in (STREET_COL, BARRIO_COL)
}).unstack(0)

In [ ]:
import matplotlib.pyplot as plt

from urban_accessibility_ml.paletas import ACC
from urban_accessibility_ml.plots import save_figure, use_style

use_style()

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for col, city in enumerate(CITIES):
    city_df = df[df[CITY_COL] == city]

    # Row 1: how many streets have 1, 2, 3... tramos
    street_sizes = city_df[STREET_COL].value_counts().value_counts().sort_index()
    ax = axes[0, col]
    ax.bar(street_sizes.index, street_sizes.values, color=ACC["Accesible"], width=0.6)
    ax.set_xticks(street_sizes.index)
    ax.set_title(f"{city}: tramos por calle", fontsize=9)
    ax.set_xlabel("Tramos de la calle")
    ax.set_ylabel("Número de calles")

    # Row 2: tramos of each barrio, from the largest to the smallest
    barrio_sizes = city_df[BARRIO_COL].value_counts()
    ax = axes[1, col]
    ax.bar(range(len(barrio_sizes)), barrio_sizes.values, color=ACC["Practicable"], width=0.8)
    ax.set_xticks([])
    ax.set_title(f"{city}: tramos por barrio ({len(barrio_sizes)} barrios)", fontsize=9)
    ax.set_xlabel("Barrios, de mayor a menor")
    ax.set_ylabel("Tramos del barrio")
fig.tight_layout()
save_figure("tramos_por_calle_y_barrio", "0.07-particion", fig=fig)
plt.show()

Las calles tienen habitualmente un solo tramo en ambas ciudades. En algunos casos tienen 2 tramos, y hay muy pocos casos con calles con 3 o 4 tramos. Llevarse la misma calle al conjunto de test o train no debería dar problemas.

Sin embargo, los barrios están más desbalanceados, puesto que hay algún barrio que tiene más tramos que el resto, y que podría hacer que los folds estuvieran más desbalanceados.

## Relación entre tramos de una misma calle
Para ver si los tramos de una calle son similares, vamos a comparar la puntuación total de cada tramo con la de otro tramo:
* Con otro tramo de su misma calle (si es que la calle tiene más de un tramo).
* Con otro tramo aleatorio de la misma ciudad.

Queremos comprobar así si existe alguna relación en los tramos de la misma calle, mirando las medias y medianas de las diferencias entre los dos casos.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from urban_accessibility_ml.paletas import ACC
from urban_accessibility_ml.plots import save_figure, use_style

use_style()
SCORE = "puntuacion_total"
rng = np.random.default_rng(0)

pairs = []  # one row per tramo of a street with several tramos: its score, a same-street tramo and a random one
for city in CITIES:
    city_df = df[df[CITY_COL] == city].reset_index(drop=True)
    for _, street in city_df.groupby(STREET_COL):
        if len(street) < 2:
            continue
        for i, score in zip(street.index, street[SCORE]):
            same = rng.choice(street.index.difference([i]))
            pairs.append((city, score, city_df.at[same, SCORE], city_df[SCORE].iloc[rng.integers(len(city_df))]))
pairs = pd.DataFrame(pairs, columns=[CITY_COL, "tramo", "misma_calle", "al_azar"])
pairs.groupby(CITY_COL).size().to_frame("tramos en calles con varios tramos")

In [ ]:
differences = pd.DataFrame({
    CITY_COL: pairs[CITY_COL],
    "misma calle": (pairs["tramo"] - pairs["misma_calle"]).abs(),
    "al azar": (pairs["tramo"] - pairs["al_azar"]).abs(),
})
differences.groupby(CITY_COL).agg(["mean", "median"]).round(3)

In [ ]:
EDGE = "#4A4D5C"  # grey of the axes: visible on light and dark backgrounds (the default black is not)
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for ax, city in zip(axes, CITIES):
    city_differences = differences[differences[CITY_COL] == city]
    box = ax.boxplot([city_differences["misma calle"], city_differences["al azar"]], tick_labels=["misma calle", "al azar"], patch_artist=True, widths=0.5,
                      whiskerprops={"color": EDGE}, capprops={"color": EDGE},
                      flierprops={"marker": "o", "markerfacecolor": EDGE, "markeredgecolor": EDGE, "markersize": 4})
    for patch, color in zip(box["boxes"], [ACC["Accesible"], ACC["Sin datos"]]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(city)
axes[0].set_ylabel("Diferencia de puntuación total\nentre los dos tramos")
fig.tight_layout()
save_figure("diferencia_misma_calle_vs_azar", "0.07-particion", fig=fig)
plt.show()

Vemos que la relación existe, aunque no es muy fuerte. Hay diferencias entre las medias y medianas en ambas ciudades, siendo menor cuando se compara con la misma calle. En Albacete la media es 0.104 frente a 0.13 aleatorio, mientras que la mediana es 0.100 respecto a 0.120. En Algeciras los resultados son similares: 0.144 frente a 0.16 aleatorio en la media, e igual resultado en la mediana.


## Tramos de clase A por pliegue
Contamos los tramos de clase A que hay en el test de cada pliegue, agrupando por calle y por barrio.

In [ ]:
def a_per_fold(splits) -> list[int]:
    return [int((class_of[s.test_ids] == "A").sum()) for s in splits]


rows = {}
for group_col in (STREET_COL, BARRIO_COL):
    for city in CITIES:
        rows[(group_col, city)] = a_per_fold(split_within_city(df, city, group_col))
    rows[(group_col, "mixto")] = a_per_fold(split_mixed(df, group_col))

table = pd.DataFrame(rows, index=[f"pliegue {i}" for i in range(5)]).T
table.columns.name = None
table.index.names = ["group_col", "estrategia"]
table["total"] = table.sum(axis=1)
table["min"] = table.drop(columns="total").min(axis=1)
table

Mostramos el tamaño del conjunto de test en cada tramo.

In [ ]:
sizes = {}
for group_col in (STREET_COL, BARRIO_COL):
    for city in CITIES:
        sizes[(group_col, city)] = [len(s.test_ids) for s in split_within_city(df, city, group_col)]
    sizes[(group_col, "mixto")] = [len(s.test_ids) for s in split_mixed(df, group_col)]

pd.DataFrame(sizes, index=[f"pliegue {i}" for i in range(5)]).T

## Conclusión
* Hemos comprobado que los tramos de una misma calle se parecen más entre sí que entre el resto de calles, aunque de una manera bastante moderada. Se elige continuar con la agrupación de tramos de la misma calle en el conjunto de entrenamiento o test.
* **Por calle**, todos los pliegues de test contienen al menos dos tramos de clase A, en las dos ciudades y en el mixto, y los tamaños de test son parecidos entre pliegues. El número de tramos por cada calle no es muy elevado (como mucho 4), así que las restricciones son mínimas a la hora de crear los pliegues.
* **Por barrio**, la cosa es diferente, en primer lugar, hay algunos pliegues que no tienen un solo tramo de clase A, sobre todo en Albacete. Además, hay bastante variación en el tamaño del conjunto de test, pasando de 55 en Algeciras como mínimo, a 166 como máximo.
* Vamos a **descartar el barrio** y los experimentos usan `nombre_calle_base`.
* A tener en cuenta durante el entrenamiento, es que la clase A es muy minoritaria, incluso sin agrupar por calle.